In [1]:

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Install Libraries

In [4]:
!pip install -q diffusers          # The library that runs Stable Diffusion
!pip install -q transformers       # Hugging Face's core library (used by both SD and BLIP)
!pip install -q accelerate         # Makes models run faster on GPU
!pip install -q fastapi            # A simple web server — this is our "API"
!pip install -q uvicorn            # Runs the FastAPI server
!pip install -q pyngrok            # Python wrapper for ngrok (the tunnel tool)
!pip install -q Pillow             # PIL — Python Image Library, for basic image editing
!pip install -q python-multipart   # Lets FastAPI receive uploaded images
print("all installations successful")

all installations successful


* diffusers: library that runs stable diffusion
* transformers: Hugging Face's core library (used by both SD and BLIP)
* accelerate: increases speed of gpu inference
* fastapi: simple web server
* uvicorn: runs fastapi server
* pyngrok: python wrapper for ngrok
* python-multipart: lets fastapi get images

# imports


In [16]:
import torch
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
from transformers import BlipProcessor, BlipForConditionalGeneration, CLIPImageProcessor
from PIL import Image
import io, base64
print("imports successful")

imports successful


* diffusers is Hugging Face's library for image generation models Pipeline = a complete ready-to-use model with all steps pre-connected StableDiffusionPipeline = takes text → makes image StableDiffusionImg2ImgPipeline = takes image + text → makes new image
* Blip is a model that sees image and geneerates text, Blipprocessor prepares image before feeding it to model, BlipForConditionalGeneration is the model that  generate text for image
* io = handles file-like objects in memory (like a fake file), base64 = converts images to text format so we can send them over internet

# Loading required models

In [9]:
#txt2img pipeline
txt2img_pipe= StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
torch_dtype= torch.float16,
safety_checker= None
)

txt2img_pipe= txt2img_pipe.to("cuda")
print("Stable Diffusion model for text to image is successfully loaded..")




Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

Stable Diffusion model for text to image is successfully loaded..


In [19]:
#img2img pipeline
img2img_pipe= StableDiffusionImg2ImgPipeline(
    vae= txt2img_pipe.vae,
    text_encoder= txt2img_pipe.text_encoder,
    tokenizer= txt2img_pipe.tokenizer,
    unet= txt2img_pipe.unet,
    scheduler= txt2img_pipe.scheduler,
    safety_checker= None,
    feature_extractor=CLIPImageProcessor.from_pretrained(
        "runwayml/stable-diffusion-v1-5", 
        subfolder="feature_extractor"
    )
).to("cuda")

print("img2img model is also ready")

blip_processor= BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
blip_model= BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype= torch.float16
).to("cuda")

print("all models ready")

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_img2img.StableDiffusionImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


img2img model is also ready


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

all models ready


# Create FastAPI Server

In [20]:
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import JSONResponse
import uvicorn
from pyngrok import ngrok

In [ ]:
#converts PIL image to base64 string 
def img2base64(pil_image):
    buffer = io.BytesIO()           # Create an empty "file" in memory (not on disk)
    pil_image.save(buffer, format="PNG")  # Save the image INTO that memory file
    buffer.seek(0)                  # Go back to the start of the memory file
    img_bytes = buffer.read()       # Read all the bytes
    img_b64 = base64.b64encode(img_bytes)     # Convert bytes to base64
    return img_b64.decode("utf-8")  # Convert bytes to a regular string


#style presets
STYLE_PRESETS = {
    "Realistic":  "",   # No addition = default realistic
    "Anime":      ", anime style, vibrant colors, Studio Ghibli",
    "Oil Painting": ", oil painting, thick brush strokes, impressionist",
    "Watercolor": ", watercolor painting, soft edges, pastel colors",
    "Cyberpunk":  ", cyberpunk, neon lights, dark city, futuristic",
    "Sketch":     ", pencil sketch, black and white, hand drawn",
}

#creating fastapi app
app= FastAPI()
# @app.post("/generate") means: when someone sends a POST request to /generate, run this function
@app.post("/generate")
async def generate_image(
    prompt: str = Form(...),        # The text the user typed
    style: str = Form("Realistic"), # Which style they chose (default: Realistic)
    steps: int = Form(20),          # How many denoising steps (more = better quality but slower)
    batch: int = Form(1)            # How many images to generate
):
    # Build the final prompt by adding style text
    style_text = STYLE_PRESETS.get(style, "")  
    # .get(key, default) = get value from dictionary, or use default if key not found
    full_prompt = prompt + style_text

    
    negative_prompt = "blurry, bad quality, ugly, deformed, low resolution"

    results=[]


    for i in range(batch):
        output= txt2img_pipe(
            prompt= full_prompt,
            negative_prompt= negative_prompt,
            num_inference_steps= steps,
            guidance_scale= 7.5,
            height= 512,
            width= 512
        )

        img= output_images[0]
        results.append(img2base64(img))

    # Return a JSON response — JSON is just a structured text format
    # Streamlit will receive this JSON and decode the images
    return JSONResponse({"images": results, "prompt_used": full_prompt})

# Image to Image endpoint
@app.post("/img2img")
async def img_to_img(
    file: UploadFile = File(...),   # The uploaded image file
    prompt: str = Form(...),        # Style description
    strength: float = Form(0.7)     # 0.0 = keep original, 1.0 = ignore original completely
):
    # Read the uploaded image bytes and convert to PIL
    image_bytes = await file.read()  # await = wait for the file to fully upload
    init_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    # .convert("RGB") = make sure it's RGB (not RGBA which has transparency)
    
    # Resize to 512x512 (SD works best at this size)
    init_image = init_image.resize((512, 512))
    
    output = img2img_pipe(
        prompt=prompt,
        image=init_image,
        strength=strength,   # How much to change the original image
        num_inference_steps=30,
        guidance_scale=7.5,
    )
    
    result_img = output.images[0]
    return JSONResponse({"image": image_to_base64(result_img)})

# Image captioning endpoint
@app.post("/caption")
async def caption_image(file: UploadFile = File(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    
    # BlipProcessor prepares the image (resizes, normalizes pixel values)
    # return_tensors="pt" means "return PyTorch tensors" (the format the model expects)
    inputs = blip_processor(image, return_tensors="pt").to("cuda", torch.float16)
    
    # Generate the caption
    # max_new_tokens=50 = generate at most 50 words
    output = blip_model.generate(**inputs, max_new_tokens=50)
    
    # Decode converts the model's number output back to text
    caption = blip_processor.decode(output[0], skip_special_tokens=True)
    # skip_special_tokens=True = remove internal tokens like [SEP], [CLS] etc.
    
    return JSONResponse({"caption": caption})

# Create Ngrok Tunnel

In [ ]:
import threading 

ngrok.set_auth_token("3D8G19KdUoEnOs9hmkpcQqzGvgr_33HWxxb6jJGU8ngE1hNcp")


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(3)  

public_url = ngrok.connect(8000)
print(f"Your API is live at: {public_url}")
print("Copy this URL — you'll paste it into your Streamlit app!")